In [1]:
from pyspark.sql import SparkSession, functions as F
import random

spark = SparkSession.builder \
    .appName("BigData") \
    .master("yarn") \
    .config("spark.sql.catalogImplementation", "hive") \
    .config("spark.hadoop.fs.defaultFS", "hdfs://namenode:9000") \
    .enableHiveSupport() \
    .getOrCreate()

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/05/15 22:39:55 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
26/05/15 22:39:59 WARN Utils: Service 'SparkUI' could not bind on port 4040. Attempting port 4041.
26/05/15 22:40:03 WARN Client: Neither spark.yarn.jars nor spark.yarn.archive is set, falling back to uploading libraries under SPARK_HOME.


In [1]:
from pyspark.sql import SparkSession, functions as F
import random

spark = SparkSession.builder \
    .appName("BigData") \
    .master("yarn") \
    .config("spark.sql.catalogImplementation", "hive") \
    .config("spark.hadoop.fs.defaultFS", "hdfs://namenode:9000") \
    .config("spark.yarn.jars", "local:/opt/spark/jars/*") \
    .enableHiveSupport() \
    .getOrCreate()

regions = ["Nord", "Sud", "Est", "Ouest", "Centre"]
produits = ["Laptop Pro", "Ecran 4K", "Clavier Mecanique", "Souris Sans Fil", "Casque Audio"]

data = [(i, random.choice(regions), random.choice(produits), 
         random.randint(1,5), float(random.randint(20,1200))) 
        for i in range(1000)]

df = spark.createDataFrame(data, ["transaction_id","region","produit","quantite","prix_unitaire"])
df = df.withColumn("chiffre_affaires", F.col("quantite") * F.col("prix_unitaire"))
df = df.dropDuplicates().dropna()

dim_produits = df.select("produit").distinct().withColumn("produit_id", F.monotonically_increasing_id())
dim_geo = df.select("region").distinct().withColumn("region_id", F.monotonically_increasing_id())
fact_ventes = df.select("transaction_id","produit","region","quantite","prix_unitaire","chiffre_affaires")

dim_produits.write.mode("overwrite").saveAsTable("dim_produits")
dim_geo.write.mode("overwrite").saveAsTable("dim_geographie")
fact_ventes.write.mode("overwrite").saveAsTable("fact_ventes")

spark.sql("SHOW TABLES").show()
print("Done!")

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/05/15 23:00:59 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
26/05/15 23:01:04 WARN Utils: Service 'SparkUI' could not bind on port 4040. Attempting port 4041.
26/05/15 23:01:17 ERROR YarnClientSchedulerBackend: The YARN application has already ended! It might have been killed or the Application Master may have failed to start. Check the YARN application logs for more details.
26/05/15 23:01:17 ERROR SparkContext: Error initializing SparkContext.
org.apache.spark.SparkException: Application application_1778884416111_0002 failed 2 times due to AM Container for appattempt_1778884416111_0002_000002 exited with  exitCode: 1
Failing this attempt.Diagnostics: [2026-05-15 23:01:16.762]Exception from container-launch.
Container id: container_e01_1778884416111_0002_02_000001
Exit code: 

Py4JJavaError: An error occurred while calling None.org.apache.spark.api.java.JavaSparkContext.
: org.apache.spark.SparkException: Application application_1778884416111_0002 failed 2 times due to AM Container for appattempt_1778884416111_0002_000002 exited with  exitCode: 1
Failing this attempt.Diagnostics: [2026-05-15 23:01:16.762]Exception from container-launch.
Container id: container_e01_1778884416111_0002_02_000001
Exit code: 1

[2026-05-15 23:01:16.792]Container exited with a non-zero exit code 1. Error file: prelaunch.err.
Last 4096 bytes of prelaunch.err :
Last 4096 bytes of stderr :
Error: Could not find or load main class org.apache.spark.deploy.yarn.ExecutorLauncher


[2026-05-15 23:01:16.793]Container exited with a non-zero exit code 1. Error file: prelaunch.err.
Last 4096 bytes of prelaunch.err :
Last 4096 bytes of stderr :
Error: Could not find or load main class org.apache.spark.deploy.yarn.ExecutorLauncher


For more detailed output, check the application tracking page:

In [2]:
spark = SparkSession.builder \
    .appName("BigData") \
    .master("local[*]") \
    .config("spark.sql.catalogImplementation", "hive") \
    .enableHiveSupport() \
    .getOrCreate()

26/05/15 23:02:04 WARN SparkContext: Another SparkContext is being constructed (or threw an exception in its constructor). This may indicate an error, since only one SparkContext should be running in this JVM (see SPARK-2243). The other SparkContext was created at:
org.apache.spark.api.java.JavaSparkContext.<init>(JavaSparkContext.scala:58)
java.base/jdk.internal.reflect.NativeConstructorAccessorImpl.newInstance0(Native Method)
java.base/jdk.internal.reflect.NativeConstructorAccessorImpl.newInstance(NativeConstructorAccessorImpl.java:62)
java.base/jdk.internal.reflect.DelegatingConstructorAccessorImpl.newInstance(DelegatingConstructorAccessorImpl.java:45)
java.base/java.lang.reflect.Constructor.newInstance(Constructor.java:490)
py4j.reflection.MethodInvoker.invoke(MethodInvoker.java:247)
py4j.reflection.ReflectionEngine.invoke(ReflectionEngine.java:374)
py4j.Gateway.invoke(Gateway.java:238)
py4j.commands.ConstructorCommand.invokeConstructor(ConstructorCommand.java:80)
py4j.commands.Con

In [1]:
26/05/15 23:02:04 WARN SparkContext: Another SparkContext is being constructed (or threw an exception in its constructor). This may indicate an error, since only one SparkContext should be running in this JVM (see SPARK-2243). The other SparkContext was created at:
org.apache.spark.api.java.JavaSparkContext.<init>(JavaSparkContext.scala:58)
java.base/jdk.internal.reflect.NativeConstructorAccessorImpl.newInstance0(Native Method)
java.base/jdk.internal.reflect.NativeConstructorAccessorImpl.newInstance(NativeConstructorAccessorImpl.java:62)
java.base/jdk.internal.reflect.DelegatingConstructorAccessorImpl.newInstance(DelegatingConstructorAccessorImpl.java:45)
java.base/java.lang.reflect.Constructor.newInstance(Constructor.java:490)
py4j.reflection.MethodInvoker.invoke(MethodInvoker.java:247)
py4j.reflection.ReflectionEngine.invoke(ReflectionEngine.java:374)
py4j.Gateway.invoke(Gateway.java:238)
py4j.commands.ConstructorCommand.invokeConstructor(ConstructorCommand.java:80)
py4j.commands.ConstructorCommand.execute(ConstructorCommand.java:69)
py4j.ClientServerConnection.waitForCommands(ClientServerConnection.java:182)
py4j.ClientServerConnection.run(ClientServerConnection.java:106)
java.base/java.lang.Thread.run(Thread.java:829)
26/05/15 23:02:04 WARN Utils: Service 'SparkUI' could not bind on port 4040. Attempting port 4041.

SyntaxError: leading zeros in decimal integer literals are not permitted; use an 0o prefix for octal integers (2856070786.py, line 1)

In [2]:
spark = SparkSession.builder \
    .appName("BigData") \
    .master("local[*]") \
    .config("spark.sql.catalogImplementation", "hive") \
    .enableHiveSupport() \
    .getOrCreate()

NameError: name 'SparkSession' is not defined

In [3]:
from pyspark.sql import SparkSession, functions as F
import random

spark = SparkSession.builder \
    .appName("BigData") \
    .master("local[*]") \
    .config("spark.sql.catalogImplementation", "hive") \
    .enableHiveSupport() \
    .getOrCreate()

regions = ["Nord", "Sud", "Est", "Ouest", "Centre"]
produits = ["Laptop Pro", "Ecran 4K", "Clavier Mecanique", "Souris Sans Fil", "Casque Audio"]

data = [(i, random.choice(regions), random.choice(produits), 
         random.randint(1,5), float(random.randint(20,1200))) 
        for i in range(1000)]

df = spark.createDataFrame(data, ["transaction_id","region","produit","quantite","prix_unitaire"])
df = df.withColumn("chiffre_affaires", F.col("quantite") * F.col("prix_unitaire"))
df = df.dropDuplicates().dropna()

dim_produits = df.select("produit").distinct().withColumn("produit_id", F.monotonically_increasing_id())
dim_geo = df.select("region").distinct().withColumn("region_id", F.monotonically_increasing_id())
fact_ventes = df.select("transaction_id","produit","region","quantite","prix_unitaire","chiffre_affaires")

dim_produits.write.mode("overwrite").saveAsTable("dim_produits")
dim_geo.write.mode("overwrite").saveAsTable("dim_geographie")
fact_ventes.write.mode("overwrite").saveAsTable("fact_ventes")

spark.sql("SHOW TABLES").show()
print("Done!")

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/05/15 23:03:24 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
26/05/15 23:03:25 WARN Utils: Service 'SparkUI' could not bind on port 4040. Attempting port 4041.
26/05/15 23:03:32 WARN HiveConf: HiveConf of name hive.stats.jdbc.timeout does not exist
26/05/15 23:03:32 WARN HiveConf: HiveConf of name hive.stats.retries.wait does not exist
26/05/15 23:03:35 WARN ObjectStore: Version information not found in metastore. hive.metastore.schema.verification is not enabled so recording the schema version 2.3.0
26/05/15 23:03:35 WARN ObjectStore: setMetaStoreSchemaVersion called but recording version is disabled: version = 2.3.0, comment = Set by MetaStore UNKNOWN@172.18.0.7
26/05/15 23:03:36 WARN HadoopFSUtils: The directory file:/opt/spark/work-dir/spark-warehouse/dim_produits was not f

+---------+--------------+-----------+
|namespace|     tableName|isTemporary|
+---------+--------------+-----------+
|  default|dim_geographie|      false|
|  default|  dim_produits|      false|
|  default|   fact_ventes|      false|
+---------+--------------+-----------+

Done!


In [1]:
spark.sql("SELECT produit, SUM(chiffre_affaires) as ca FROM fact_ventes GROUP BY produit ORDER BY ca DESC LIMIT 10").toPandas().to_csv("/notebooks/top10.csv", index=False)
spark.sql("SELECT region, SUM(chiffre_affaires) as ca FROM fact_ventes GROUP BY region").toPandas().to_csv("/notebooks/ventes_region.csv", index=False)

NameError: name 'spark' is not defined

In [2]:
from pyspark.sql import SparkSession, functions as F
import random

spark = SparkSession.builder \
    .appName("BigData") \
    .master("local[*]") \
    .config("spark.sql.catalogImplementation", "hive") \
    .enableHiveSupport() \
    .getOrCreate()

spark.sql("SELECT produit, SUM(chiffre_affaires) as ca FROM fact_ventes GROUP BY produit ORDER BY ca DESC LIMIT 10").toPandas().to_csv("/notebooks/top10.csv", index=False)
spark.sql("SELECT region, SUM(chiffre_affaires) as ca FROM fact_ventes GROUP BY region").toPandas().to_csv("/notebooks/ventes_region.csv", index=False)
print("Done")

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/05/15 23:44:31 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
26/05/15 23:44:33 WARN Utils: Service 'SparkUI' could not bind on port 4040. Attempting port 4041.
26/05/15 23:44:39 WARN HiveConf: HiveConf of name hive.stats.jdbc.timeout does not exist
26/05/15 23:44:39 WARN HiveConf: HiveConf of name hive.stats.retries.wait does not exist
26/05/15 23:44:42 WARN ObjectStore: Version information not found in metastore. hive.metastore.schema.verification is not enabled so recording the schema version 2.3.0
26/05/15 23:44:42 WARN ObjectStore: setMetaStoreSchemaVersion called but recording version is disabled: version = 2.3.0, comment = Set by MetaStore UNKNOWN@172.18.0.7


ImportError: Pandas >= 1.0.5 must be installed; however, it was not found.

In [3]:
import subprocess
subprocess.run(["pip", "install", "pandas"])

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.8/12.8 MB 5.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.8/16.8 MB 2.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 510.1/510.1 KB 3.1 MB/s eta 0:00:00


CompletedProcess(args=['pip', 'install', 'pandas'], returncode=0)

In [4]:
from pyspark.sql import SparkSession, functions as F
import random

spark = SparkSession.builder \
    .appName("BigData") \
    .master("local[*]") \
    .config("spark.sql.catalogImplementation", "hive") \
    .enableHiveSupport() \
    .getOrCreate()

spark.sql("SELECT produit, SUM(chiffre_affaires) as ca FROM fact_ventes GROUP BY produit ORDER BY ca DESC LIMIT 10").toPandas().to_csv("/notebooks/top10.csv", index=False)
spark.sql("SELECT region, SUM(chiffre_affaires) as ca FROM fact_ventes GROUP BY region").toPandas().to_csv("/notebooks/ventes_region.csv", index=False)
print("Done")

Done
